# 05과 - 에이전틱 RAG


## 설정

이 노트북은 Microsoft Agent Framework를 사용하여 Agentic RAG(Retrieval-Augmented Generation) 패턴을 시연합니다.

**필수 조건:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI 검색 서비스 엔드포인트
- `AZURE_SEARCH_API_KEY` — Azure AI 검색 API 키
- 환경 변수를 통해 구성된 Azure OpenAI 배포
- Azure CLI 인증(`az login`) 완료


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG란?

전통적인 RAG는 고정된 파이프라인을 따릅니다: 문서를 검색한 후 응답을 생성합니다. **Agentic RAG**는 에이전트에게 정보를 **언제** 그리고 **어떻게** 검색할지 자율성을 부여함으로써 한 단계 더 나아갑니다.

Agentic RAG를 사용하면 에이전트는:
- 질문에 답하기 전에 검색이 필요한지 **결정**할 수 있고
- 어떤 데이터 소스나 도구를 조회할지 **선택**할 수 있으며
- 검색된 결과를 **평가**하고 첫 시도가 충분하지 않을 경우 후속 검색을 수행할 수 있습니다
- 여러 검색 단계를 통해 얻은 정보를 결합하여 일관된 답변을 **작성**할 수 있습니다

이로 인해 에이전트는 고정된 검색 후 생성 파이프라인에 비해 더 유연하고 정확해집니다.


## 검색 도구 만들기

Agentic RAG에서 외부 데이터 소스는 에이전트가 필요에 따라 호출할 수 있는 **도구**로 래핑됩니다. 이를 통해 에이전트는 검색을 필수 단계가 아닌 수행할 수 있는 또 다른 작업으로 간주할 수 있습니다.

아래에서는 여행 지식 베이스를 정의하고 에이전트가 목적지 정보를 조회할 수 있도록 도구로 노출합니다.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG 에이전트 구축

이제 **항상 대답하기 전에 정보를 검색하도록** 지시된 에이전트를 만듭니다. 이 에이전트는 자체 학습 데이터에 의존하지 않고 지식 기반에 기반한 응답을 하기 위해 `search_travel_knowledge` 도구를 사용합니다.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 반복 검색 — 메이커-체커 패턴

Agentic RAG의 주요 장점은 **반복 검색**입니다. 에이전트는 초기 결과를 확인, 수정 또는 확장하기 위해 여러 차례 검색을 수행할 수 있습니다 — "메이커-체커" 작업 흐름과 유사하게:

1. **메이커 단계**: 에이전트가 초기 정보를 검색하고 응답 초안을 작성합니다.
2. **체커 단계**: 에이전트가 세부 사항을 확인하거나 빈틈을 채우기 위해 추가 검색을 수행합니다.

아래에서는 여러 목적지를 비교해야 하는 질문이 에이전트에게 주어져 여러 번 검색하도록 유도합니다.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 요약

이 강의에서는 Microsoft Agent Framework를 사용하여 **Agentic RAG** 시스템을 구축하는 방법을 배웠습니다:

- **Agentic RAG**는 에이전트가 정보 검색 시점을 자율적으로 결정할 수 있게 하여, 고정된 검색이 아닌 동적 검색을 가능하게 합니다.
- **도구를 데이터 소스로 활용**: 외부 지식 베이스(예: Azure AI Search)를 에이전트가 호출할 수 있는 도구로 래핑합니다.
- **반복적 검색**: 메이커-체커 패턴을 통해 에이전트가 최종 답변을 생성하기 전에 여러 차례 검색, 검증, 정제를 수행할 수 있습니다.

실제 환경에서는 대규모 여행 문서 검색을 처리하기 위해 인메모리 `TRAVEL_KNOWLEDGE_BASE` 대신 실제 Azure AI Search 인덱스를 사용하게 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 저희는 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확성이 포함될 수 있음을 양지해 주시기 바랍니다. 원본 문서의 원어 자료가 권위 있는 출처로 간주되어야 합니다. 중요한 정보에 대해서는 전문적인 인간 번역을 권장합니다. 이 번역의 사용으로 인해 발생하는 어떠한 오해나 잘못된 해석에 대해서도 저희는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
